# Evaluación de TopSim, P-Score y espectro singular en `val_4cases`

Este notebook calcula 3 métricas de representación sobre los 6 datasets del repo en el split `val_4cases`,
comparando dos backbones:

1. **ResNet-18 random init** (`weights=None`)
2. **ResNet-18 preentrenada en ImageNet** (`ResNet18_Weights.IMAGENET1K_V1`)

Métricas:
- **Topographic similarity (TopSim)** entre targets semánticos y embeddings.
- **Parallelism score (P-score)** por atributo (promedio sobre atributos).
- **Espectro singular** y número mínimo de componentes para explicar `X%` de varianza (`X=90%` por defecto).


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights, resnet18

sys.path.append("..")

from visgen.datasets import (
    Cars3D,
    CLEVR,
    DSprites,
    IRAVEN,
    MPI3D,
    Shapes3D,
    make_validation_subset as split_make_validation_subset,
)
from visgen.datasets.non_iid import subset_with_four_case_support
from visgen.models.metrics import (
    n_components_for_variance,
    parallelism_score_categorical,
    singular_value_report,
    topographic_similarity,
)
from visgen.utils.general import register_resolvers

register_resolvers()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


In [14]:
DATASET_CONFIG_DIR = Path("../configs/datasets")
DATASETS_BASE_PATH = Path("../data")
VARIANCE_THRESHOLD = 0.90
MAX_SAMPLES_PER_DATASET = 3000  # ajusta si necesitas más precisión
BATCH_SIZE = 128
NUM_WORKERS = 0
SEED = 0

DATASET_CLASSES = {
    "dsprites": DSprites,
    "mpi3d": MPI3D,
    "shapes3d": Shapes3D,
    "cars3d": Cars3D,
    "iraven": IRAVEN,
    "clevr": CLEVR,
}


def load_training_cfg(name, datasets_base_path=DATASETS_BASE_PATH):
    cfg = OmegaConf.load(DATASET_CONFIG_DIR / f"{name}.yml").data.training
    raw_path = Path(cfg.path)
    if not raw_path.is_absolute():
        try:
            relative_to_data = raw_path.relative_to("data")
        except ValueError:
            relative_to_data = raw_path
        cfg.path = str(Path(datasets_base_path) / relative_to_data)
    return cfg


def make_val_4cases_subset(name, seed=SEED):
    cfg = load_training_cfg(name)
    dataset = DATASET_CLASSES[name](**cfg)
    _, val_data = split_make_validation_subset(
        dataset,
        val_fraction=cfg.val_fraction,
        seed=seed,
        num_ood_val=cfg.num_ood_val if "num_ood_val" in cfg else 1,
    )

    allowed_attributes = list(cfg.targets) if cfg.targets else [attr.name for attr in cfg.attributes]
    shared_other_attributes = True
    if "non_iid" in cfg and cfg.non_iid is not None and not isinstance(cfg.non_iid, str):
        shared_other_attributes = cfg.non_iid.get("shared_other_attributes", True)

    val_4cases = subset_with_four_case_support(
        val_data,
        allowed_attributes=allowed_attributes,
        shared_other_attributes=shared_other_attributes,
    )
    return val_4cases


In [9]:
class ResNet18Embedding(torch.nn.Module):
    def __init__(self, pretrained: bool):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = resnet18(weights=weights)
        self.backbone = torch.nn.Sequential(*list(model.children())[:-1])

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.backbone(x)
        return z.flatten(1)


def _adapt_channels(x: torch.Tensor) -> torch.Tensor:
    # ResNet de torchvision espera 3 canales.
    if x.ndim == 3:
        x = x.unsqueeze(1)
    if x.shape[1] == 1:
        x = x.repeat(1, 3, 1, 1)
    return x


@torch.no_grad()
def extract_embeddings(dataset, model, max_samples=MAX_SAMPLES_PER_DATASET):
    def _run_with_workers(num_workers: int):
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)
        zs, ys = [], []
        n = 0
        for x, y in loader:
            if n >= max_samples:
                break
            remaining = max_samples - n
            x = x[:remaining]
            y = y[:remaining]
            x = _adapt_channels(x).to(device)
            z = model(x).cpu().numpy()
            zs.append(z)
            ys.append(np.asarray(y))
            n += len(x)
        return zs, ys

    try:
        zs, ys = _run_with_workers(NUM_WORKERS)
    except OSError as exc:
        if NUM_WORKERS == 0:
            raise
        print(f"[warn] DataLoader con num_workers={NUM_WORKERS} falló ({exc}). Reintentando con num_workers=0.")
        zs, ys = _run_with_workers(0)

    if not zs:
        raise RuntimeError("No se pudieron extraer embeddings (dataset vacío o sin lectura).")

    z = np.concatenate(zs, axis=0)
    y = np.concatenate(ys, axis=0)
    if y.ndim == 1:
        y = y[:, None]
    return z, y


In [10]:
def compute_representation_metrics(z: np.ndarray, y: np.ndarray, variance_threshold=VARIANCE_THRESHOLD):
    # TopSim: distancia coseno en semántica (targets) y embeddings.
    topsim = topographic_similarity(
        semantic_representations=y,
        observed_representations=z,
        semantic_metric="cosine",
        observed_metric="cosine",
    )

    # P-score por atributo: para cada atributo, contexto = resto de atributos.
    ps_attr = []
    for attr_idx in range(y.shape[1]):
        attr = y[:, attr_idx]
        if y.shape[1] == 1:
            # sin contexto, no es definible
            continue
        context = np.delete(y, attr_idx, axis=1)
        score = parallelism_score_categorical(
            representations=z,
            attribute=attr,
            contexts=context,
        )
        if not np.isnan(score):
            ps_attr.append(score)

    pscore_mean = float(np.mean(ps_attr)) if ps_attr else np.nan

    svd = singular_value_report(z)
    n_for_x = n_components_for_variance(z, variance_threshold=variance_threshold)

    return {
        "topsim": topsim,
        "pscore_mean": pscore_mean,
        "n_components_for_threshold": n_for_x,
        "variance_threshold": variance_threshold,
        "sv_explained_ratio": svd.explained_variance_ratio,
        "sv_cumulative_ratio": svd.cumulative_explained_variance_ratio,
    }


In [15]:
torch.manual_seed(SEED)
np.random.seed(SEED)
from tqdm.notebook import tqdm
models = {
    "resnet18_random": ResNet18Embedding(pretrained=False).to(device).eval(),
    "resnet18_imagenet": ResNet18Embedding(pretrained=True).to(device).eval(),
}

all_results = []
full_details = {}

for ds_name in tqdm(DATASET_CLASSES):
    print(f"\n=== Dataset: {ds_name} ===")
    val_4cases = make_val_4cases_subset(ds_name)
    print(f"val_4cases size: {len(val_4cases)}")

    for model_name, model in tqdm(models.items()):
        print(f"  -> model: {model_name}")
        z, y = extract_embeddings(val_4cases, model)
        metrics = compute_representation_metrics(z, y.squeeze())

        all_results.append(
            {
                "dataset": ds_name,
                "model": model_name,
                "num_samples": int(z.shape[0]),
                "embedding_dim": int(z.shape[1]),
                "topsim": metrics["topsim"],
                "pscore_mean": metrics["pscore_mean"],
                f"n_components_{int(metrics['variance_threshold']*100)}pct": metrics["n_components_for_threshold"],
            }
        )

        full_details[(ds_name, model_name)] = metrics

results_df = pd.DataFrame(all_results)
results_df


  0%|          | 0/6 [00:00<?, ?it/s]


=== Dataset: dsprites ===
val_4cases size: 68058


  0%|          | 0/2 [00:00<?, ?it/s]

  -> model: resnet18_random
  -> model: resnet18_imagenet

=== Dataset: mpi3d ===
val_4cases size: 87557


  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

  -> model: resnet18_random
  -> model: resnet18_imagenet

=== Dataset: clevr ===
val_4cases size: 10000


  0%|          | 0/2 [00:00<?, ?it/s]

  -> model: resnet18_random
  -> model: resnet18_imagenet


,dataset,model,num_samples,embedding_dim,topsim,pscore_mean,n_components_90pct
0,dsprites,resnet18_random,3000,512,0.405820,0.296348,42
1,dsprites,resnet18_imagenet,3000,512,0.237449,0.272564,44
2,mpi3d,resnet18_random,3000,512,0.047534,0.356797,33
3,mpi3d,resnet18_imagenet,3000,512,0.185917,0.277734,34
4,shapes3d,resnet18_random,3000,512,0.266290,0.349238,31
5,shapes3d,resnet18_imagenet,3000,512,0.087667,0.294491,46
6,cars3d,resnet18_random,1557,512,0.041394,0.215897,80
7,cars3d,resnet18_imagenet,1557,512,-0.009897,0.218497,58
8,iraven,resnet18_random,3000,512,0.403271,0.537939,1
9,iraven,resnet18_imagenet,3000,512,0.277625,0.339947,19


In [16]:
# Tabla pivote rápida para comparar modelos por dataset
pivot = results_df.pivot(index="model", columns="dataset", values=["topsim", "pscore_mean", f"n_components_{int(VARIANCE_THRESHOLD*100)}pct"])
pivot


topsim                                                    \
dataset              cars3d     clevr  dsprites    iraven     mpi3d  shapes3d   
model                                                                           
resnet18_imagenet -0.009897  0.236443  0.237449  0.277625  0.185917  0.087667   
resnet18_random    0.041394  0.060747  0.405820  0.403271  0.047534  0.266290   

                  pscore_mean                                          \
dataset                cars3d     clevr  dsprites    iraven     mpi3d   
model                                                                   
resnet18_imagenet    0.218497  0.495116  0.272564  0.339947  0.277734   
resnet18_random      0.215897  0.344859  0.296348  0.537939  0.356797   

                            n_components_90pct                              \
dataset            shapes3d             cars3d clevr dsprites iraven mpi3d   
model                                                                        
resnet18_imagenet  0.294491               58.0  39.0     44.0   19.0  34.0   
resnet18_random    0.349238               80.0  20.0     42.0    1.0  33.0   

                            
dataset           shapes3d  
model                       
resnet18_imagenet     46.0  
resnet18_random       31.0

In [ ]:
# Ejemplo: inspeccionar curva acumulada de varianza para una combinación
import matplotlib.pyplot as plt

dataset_name = "dsprites"
model_name = "resnet18_imagenet"

cum = full_details[(dataset_name, model_name)]["sv_cumulative_ratio"]
plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, len(cum)+1), cum)
plt.axhline(VARIANCE_THRESHOLD, color="red", linestyle="--", label=f"{int(VARIANCE_THRESHOLD*100)}%")
plt.xlabel("# componentes")
plt.ylabel("varianza acumulada explicada")
plt.title(f"{dataset_name} - {model_name}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
